# Attention model results — SW22 × TW48
Run in order. Only Q/Z summary CSVs, the Z-factor PNG, and the average-ratio PNG save automatically. Other save lines are commented. Original calculations and window selections are retained.


In [ ]:
from pathlib import Path
from google.colab import drive
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib import rcParams

plt.rcParams["font.family"] = "DejaVu Serif"
sw, tw = 22, 48
MODEL_TYPE = "temporal_attention"
drive.mount("/content/drive_analysis")
ATTENTION_ROOT = Path("/content/drive_analysis/MyDrive/[FuSe][STSM] Megha Babu/Attention model")
DATA_DIR = ATTENTION_ROOT / "Demeter-Anomaly-Detection-Framework" / "Data"
RESULTS_DIR = ATTENTION_ROOT / MODEL_TYPE / "outputs"
output_dir = ATTENTION_ROOT / MODEL_TYPE / "Analysis_results"
if not RESULTS_DIR.is_dir():
    raise FileNotFoundError(f"Results folder not found: {RESULTS_DIR}")
if not list(RESULTS_DIR.glob(f"result_Hp_tw{tw}-30dBG_*-{MODEL_TYPE}.csv")):
    raise FileNotFoundError(f"No final result CSVs found in {RESULTS_DIR}")
output_dir.mkdir(parents=True, exist_ok=True)
q_dir = output_dir
models = list("ABCDE") + [f"In-{c}" for c in "ABCDE"]
df_hparams = pd.DataFrame({
    "Naming": [f"tw{tw}-30dBG_{m}-{MODEL_TYPE}" for m in models],
    "tw": tw,
})

def model_label(name):
    return name.removeprefix(f"tw{tw}-30dBG_").removesuffix(f"-{MODEL_TYPE}")


In [ ]:
rst = pd.read_pickle(DATA_DIR / f"RST_30Days_Bg_SW{sw}_TW{tw}.pkl")
combined_wide = pd.DataFrame({
    "q2_All": [np.percentile(rst[tw], 50) * 100],
    "0.3_All": [np.percentile(rst[tw], 0.3) * 100],
    "99.7_All": [np.percentile(rst[tw], 99.7) * 100],
}, index=[tw])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

def plot_ratio_errorbars_combined(df, tw, category, name, outpath):
    rcParams['font.family'] = 'DejaVu Serif'
    t = combined_wide
    tr = df[df['split'].str.lower() == 'train'].sort_values('window')
    va = df[df['split'].str.lower() == 'val'].sort_values('window')
    x_tr = tr['window'].to_numpy()
    x_va = va['window'].to_numpy()
    y = float(t['q2_All'].iloc[0])
    e1 = float(t['0.3_All'].iloc[0])
    e2 = float(t['99.7_All'].iloc[0])
    if category.lower() == 'all':
        y_tr, e_tr = (tr['agg_value'].to_numpy(float), tr['agg_error'].to_numpy(float))
        y_va, e_va = (va['agg_value'].to_numpy(float), va['agg_error'].to_numpy(float))
        title_suffix = 'All data'
    else:
        y_tr, e_tr = (tr['agg_value_sc'].to_numpy(float), tr['agg_error_sc'].to_numpy(float))
        y_va, e_va = (va['agg_value_sc'].to_numpy(float), va['agg_error_sc'].to_numpy(float))
        title_suffix = 'Quiet data'
    plt.figure(figsize=(10, 5))
    m = ~np.isnan(y_tr) & ~np.isnan(e_tr)
    if m.any():
        plt.errorbar(x_tr[m], y_tr[m], yerr=e_tr[m], fmt='o', capsize=3, label='Train')
    m = ~np.isnan(y_va) & ~np.isnan(e_va)
    if m.any():
        plt.errorbar(x_va[m], y_va[m], yerr=e_va[m], fmt='o', capsize=3, label='Validation')
    plt.axhline(y, color='RED', linewidth=1.5, label='Q2 RST')
    plt.axhline(e1, color='RED', linestyle='--', linewidth=1)
    plt.axhline(e2, color='RED', linestyle='--', linewidth=1, label='$0.3–99.7\\%$ RST')
    plt.xlabel('Window', fontsize=21)
    plt.ylabel('Seismic Anomaly ratio (%)', fontsize=20)
    plt.xticks(x_tr, [f'w{int(w)}' for w in x_tr], fontsize=16, rotation=50)
    plt.xticks(fontsize=22)
    plt.yticks(fontsize=22)
    plt.ylim(20, 130)
    plt.legend(fontsize=15, loc='upper left')
    plt.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    # plt.savefig(outpath, dpi=600)
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from matplotlib import rcParams

def plot_anomaly_bars(df, category, name, outpath):
    rcParams['font.family'] = 'DejaVu Serif'
    tr = df[df['split'].str.lower() == 'train'].sort_values('window')
    va = df[df['split'].str.lower() == 'val'].sort_values('window')
    if category.lower() == 'all':
        tot_tr = tr['anomalies'].to_numpy(float)
        tot_va = va['anomalies'].to_numpy(float)
        ratio_tr = tr['agg_value'].to_numpy(float) / 100.0
        ratio_va = va['agg_value'].to_numpy(float) / 100.0
        title_suffix = 'All data'
    else:
        tot_tr = tr['anomalies_sc'].to_numpy(float)
        tot_va = va['anomalies_sc'].to_numpy(float)
        ratio_tr = tr['agg_value_sc'].to_numpy(float) / 100.0
        ratio_va = va['agg_value_sc'].to_numpy(float) / 100.0
        title_suffix = 'Quiet'
    seis_tr = tot_tr * ratio_tr
    non_tr = tot_tr - seis_tr
    seis_va = tot_va * ratio_va
    non_va = tot_va - seis_va
    x = np.arange(len(tr))
    width = 0.35
    hatch_pat = '//'
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - width / 2, seis_tr, width, color='tab:blue', edgecolor='k', hatch=hatch_pat)
    ax.bar(x - width / 2, non_tr, width, bottom=seis_tr, color='tab:blue', edgecolor='k')
    dark_orange = '#FF8C00'
    ax.bar(x + width / 2, seis_va, width, color=dark_orange, edgecolor='k', hatch=hatch_pat)
    ax.bar(x + width / 2, non_va, width, bottom=seis_va, color=dark_orange, edgecolor='k')
    ax.set_xlabel('Window', fontsize=20)
    ax.set_ylabel('Anomaly Count', fontsize=20)
    ax.set_title(f'Model {name}', fontsize=22)
    win_ticks = tr['window'].to_numpy(int)
    ax.set_xticks(x)
    ax.set_xticklabels([f'w{w}' for w in win_ticks], fontsize=18)
    ax.tick_params(axis='y', labelsize=18)
    ax.legend(handles=[Patch(facecolor='tab:blue', edgecolor='k', hatch=hatch_pat, label='Train - Seismic'), Patch(facecolor='tab:blue', edgecolor='k', label='Train - Non-seismic'), Patch(facecolor=dark_orange, edgecolor='k', hatch=hatch_pat, label='Val - Seismic'), Patch(facecolor=dark_orange, edgecolor='k', label='Val - Non-seismic')], ncol=2, frameon=False, loc='upper left', fontsize=12)
    plt.tight_layout()
    # plt.savefig(outpath, dpi=600)
    plt.show()


In [ ]:
for name in df_hparams["Naming"]:
    file = RESULTS_DIR / f"result_Hp_{name}.csv"
    if not file.is_file():
        print(f"Missing: {file.name}")
        continue
    df = pd.read_csv(file)
    if "window" not in df:
        df["window"] = np.arange(len(df)) // 2
    plot_ratio_errorbars_combined(df, tw, "all", model_label(name), output_dir / f"ratio_all-{name}.png")
    plot_anomaly_bars(df, "all", model_label(name), output_dir / f"precision_all-{name}.png")


In [ ]:
import os
import numpy as np
import pandas as pd

def z_factor(df_hparams, combined_wide, mode, output_dir):
    model_names = []
    avg_train_list, err_train_list = ([], [])
    avg_eval_list, err_eval_list = ([], [])
    tw1 = []
    z_va = []
    z_tr = []
    rho_R_all = []
    rho_R99_7_all = []
    z_va_err32 = []
    z_tr_err32 = []
    z_va_err68 = []
    z_tr_err68 = []
    for m, hp in df_hparams.iterrows():
        file_path = RESULTS_DIR / f"result_Hp_{hp['Naming']}.csv"
        if not os.path.exists(file_path):
            continue
        mode = mode.lower().strip()
        if mode not in {'all', 'quiet'}:
            raise ValueError("mode must be 'all' or 'quiet'")
        value_col = 'agg_value' if mode == 'all' else 'agg_value_sc'
        error_col = 'agg_error' if mode == 'all' else 'agg_error_sc'
        ratio_col = 'q2_All'
        ratio_col2 = '99.7_All'
        df = pd.read_csv(file_path)
        tr = df[df['split'].str.lower() == 'train'].copy()
        va = df[df['split'].str.lower() == 'val'].copy()
        if m <= 4:
            tr = tr.iloc[1:16].reset_index(drop=True)
            va = va.iloc[1:16].reset_index(drop=True)
        else:
            tr = tr.iloc[:16].reset_index(drop=True)
            va = va.iloc[:16].reset_index(drop=True)
        average_train = np.mean(tr[value_col])
        error_train = np.sqrt(np.sum(tr[error_col] ** 2)) / len(tr)
        average_val = np.mean(va[value_col])
        error_val = np.sqrt(np.sum(va[error_col] ** 2)) / len(va)
        rho_R = float(combined_wide[ratio_col].iloc[0])
        rho_R99_7 = float(combined_wide[ratio_col2].iloc[0])
        rho_i_val = va[value_col].to_numpy()
        sigma_i_val = va[error_col].to_numpy()
        rho_i_tr = tr[value_col].to_numpy()
        sigma_i_tr = tr[error_col].to_numpy()
        vals_val = (rho_i_val - rho_R) / sigma_i_val
        vals_tr = (rho_i_tr - rho_R) / sigma_i_tr
        z = np.mean(vals_val)
        z_t = np.mean(vals_tr)
        p32_val, p68_val = np.percentile(vals_val, [32, 68])
        p32_tr, p68_tr = np.percentile(vals_tr, [32, 68])
        z_va.append(z)
        z_tr.append(z_t)
        z_va_err32.append(p32_val)
        z_tr_err32.append(p32_tr)
        z_va_err68.append(p68_val)
        z_tr_err68.append(p68_tr)
        rho_R99_7_all.append(rho_R99_7)
        rho_R_all.append(rho_R)
        model_names.append(hp['Naming'])
        tw1.append(hp['tw'])
        avg_train_list.append(average_train)
        err_train_list.append(error_train)
        avg_eval_list.append(average_val)
        err_eval_list.append(error_val)
    results = pd.DataFrame({'Naming': model_names, 'RST_Median': rho_R_all, 'RST_99.7': rho_R99_7_all, f'z_va_{mode}': z_va, f'z_tr_{mode}': z_tr, f'z_tr_err32_{mode}': z_tr_err32, f'z_va_err32_{mode}': z_va_err32, f'z_tr_err68_{mode}': z_tr_err68, f'z_va_err68_{mode}': z_va_err68, f'avg_train_{mode}': avg_train_list, f'err_train_{mode}': err_train_list, f'avg_val_{mode}': avg_eval_list, f'err_val_{mode}': err_eval_list, 'mode': mode, 'tw': tw1}).reset_index(drop=True)
    os.makedirs(output_dir, exist_ok=True)
    csv_path = os.path.join(q_dir, f'Qfactor-SW{sw}x{sw}_TW{tw}-{mode}_newz.csv')
    results.to_csv(csv_path, index=False)
    return results


In [ ]:
q_factor_all = z_factor(df_hparams, combined_wide, "all", output_dir)
q_factor_quiet = z_factor(df_hparams, combined_wide, "quiet", output_dir)
display(q_factor_all)
display(q_factor_quiet)


In [ ]:
def plot_q_results_range(results_df, mode, tw, Rst, output_dir='.', sw=None):
    plt.rcParams['font.family'] = 'DejaVu Serif'
    rho_R = float(Rst['q2_All'].iloc[0])
    rho_R99_7 = float(Rst['99.7_All'].iloc[0])
    rho_R0_3 = float(Rst['0.3_All'].iloc[0])
    y_lower = results_df[f'z_va_err32_{mode}'].values
    y_upper = results_df[f'z_va_err68_{mode}'].values
    y = results_df[f'z_va_{mode}'].values
    x = np.arange(len(results_df))
    naming_raw = results_df['Naming'].astype(str).values
    naming_short = [model_label(name) for name in naming_raw]
    plt.figure(figsize=(8, 5), dpi=600)
    plt.vlines(x, y_lower, y_upper, color='darkorange', linewidth=2, label='Model Z_f 32-68%')
    plt.scatter(x, y, color='darkorange', s=35, zorder=3, label='Model Z_f ')
    plt.axhline(0.0, color='red', linestyle='-', linewidth=1.5, label='Z score = 0')
    plt.xticks(x, naming_short, rotation=0, ha='center', fontsize=11)
    plt.xlabel('Model', fontsize=17)
    plt.ylabel('Z-Factor', fontsize=17)
    z2 = 2 / np.sqrt(15)
    z3 = 3 / np.sqrt(15)
    z5 = 5 / np.sqrt(15)
    plt.axhline(z5, color='green', linestyle='--', linewidth=1, label='Z score = 5')
    plt.axhline(z2, color='Red', linestyle='--', linewidth=1, label='Z score = 2')
    plt.axhline(z3, color='red', linestyle=':', linewidth=1, label='Z score = 3')
    if sw is not None:
        title = f'Z-factor Analysis [2005-2009]'
    else:
        title = f'WOG_TW{tw}'
    plt.title(title, fontsize=20, fontweight='bold')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tick_params(axis='y', labelsize=16)
    plt.tick_params(axis='x', labelsize=16)
    plt.ylim(-3, 3)
    plt.legend(fontsize=12, frameon=False, loc='best')
    plt.tight_layout()
    fname_sw = f'SW{sw}_' if sw is not None else ''
    fname = f'Z_va_{fname_sw}TW{tw}_{mode}_ranges-poster.png'
    plt.savefig(os.path.join(output_dir, fname), dpi=600, bbox_inches='tight')
    plt.show()
plot_q_results_range(q_factor_all, 'all', tw, combined_wide, output_dir=output_dir, sw=sw)


In [ ]:
def plot_train_val_together(results_all, tw, combined_wide, output_dir):
    plt.rcParams['font.family'] = 'DejaVu Serif'
    results_all = results_all[results_all['tw'].astype(int) == int(tw)].copy()
    results_all['newNaming'] = results_all['Naming'].astype(str).map(model_label)
    q2 = float(combined_wide['q2_All'].iloc[0])
    ul = float(combined_wide['99.7_All'].iloc[0])
    u2 = float(combined_wide['0.3_All'].iloc[0])
    x = np.arange(len(results_all))
    width = 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.set_title(f'Average Seismic Anomaly Ratio [2005-2009]', fontsize=19, weight='bold')
    ax.errorbar(x - width / 2, results_all['avg_train_all'], yerr=results_all['err_train_all'], fmt='o', capsize=5, color='tab:blue', label='Training', alpha=0.9)
    ax.errorbar(x + width / 2, results_all['avg_val_all'], yerr=results_all['err_val_all'], fmt='o', capsize=5, color='tab:orange', label='Validation', alpha=0.9)
    ax.axhline(q2, color='tab:red', linestyle='-', lw=1.2, label='Q2 RST')
    ax.axhline(ul, color='tab:red', linestyle='--', lw=1.2, label='0.3-99.7% RST')
    ax.axhline(u2, color='tab:red', linestyle='--', lw=1.2)
    ax.set_xticks(x)
    ax.set_xticklabels(results_all['newNaming'], rotation=0, ha='right')
    ax.set_ylabel('Average Seismic Anomaly Ratio (%)', fontsize=17)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(fontsize=13, loc='best')
    ax.tick_params(axis='both', labelsize=16)
    ax.set_ylim(30, 100)
    plt.xlabel('Model', fontsize=17)
    fig.tight_layout()
    plt.savefig(os.path.join(output_dir, f'Average-train_val-SW{sw}_TW{tw}.png'), dpi=600)
    plt.show()
plot_train_val_together(q_factor_all, tw, combined_wide, output_dir)
